# Small deterministic example with one unit (Section 5.1 of the paper)

This notebook reproduces, in order, the numbers and claims of Section 5.1 of the paper,
each one checked by an `@assert`:

1. the instance (Table 1): **one unit**, **T = 6** one-hour periods, initially off, **deterministic** demand
   (`S = 1`, `force = 0`: the scenario tree reduces to a single path); load shedding and overproduction cost
   700 per MWh (the package constants `SHEDDING_COST` and `CURTAILEMENT_COST`);
2. the exact optimum (80 200), its schedule `[1:7]` and its production trajectory;
3. the convergence of the bound (Table 2): **2 cuts** at the root for the interval formulation, **6** for the 3-bin one;
4. the first root cuts, built at the same schedule (all off), and their comparison.

**Counting convention.** The bound `L_k` of iteration `k` is the value of the root problem with the `k-1` cuts
of the previous iterations. A formulation "reaches the optimum after `n` cuts" if `L_{n+1}` is the optimum.

In [ ]:
using MSUC, Gurobi, Printf, Random, LinearAlgebra, Plots

GRB_ENV = Gurobi.Env()
# Fixed Threads and Seed: the root MIP has multiple optimal solutions, so this makes the schedules (hence
# the iteration counts) reproducible. OutputFlag comes first, otherwise Gurobi announces every parameter
# for each model created.
solver(seed = 0) = optimizer_with_attributes(() -> Gurobi.Optimizer(GRB_ENV), "OutputFlag" => 0, "Threads" => 1, "Seed" => seed)

## The instance (Table 1)

The unit is described by its minimum and maximum outputs `Pmin` and `Pmax`, its ramp `R` (up and down),
its linear cost `c`, its fixed cost (per period on), its startup cost, and its minimum up time.
In the code `S^U = S^D = Pmin`: the unit starts up and shuts down at `Pmin`. The minimum up and down times
are 1: no restriction. The operating intervals `[a, b]` are the periods `a, ..., b-1` during which the unit is on
(`b = T+1`: until the end of the horizon).

In [ ]:
T = 6
DEMAND = [90.0, 80.0, 130.0, 60.0, 100.0, 90.0]
FORCE = 0.0                       # 0: deterministic

# pmin, pmax, ramp, linear cost, fixed cost (per period on), startup cost, minimum up time
function make_unit(name; pmin, pmax, ramp, cost, fixed, startup, minup)
    return MSUC.ThermalUnit(
        name, 1,       # name, network node
        pmin, pmax,    # minimum, maximum output
        ramp, ramp,    # ramp up, ramp down
        startup, 0.0,  # startup, shutdown cost
        cost, fixed,   # linear cost, fixed cost
        0.0, -1,       # initial output and status: off
        minup, 1,      # minimum up time, down time
        MSUC.set_intervals(minup, -1, 0.0, pmin, ramp, T),
    )
end

units = [make_unit(1; pmin = 40.0, pmax = 100.0, ramp = 30.0, cost = 20.0, fixed = 50.0, startup = 500.0, minup = 1)]
instance = MSUC.Instance("small_example", T, length(units), units, MSUC.IntervalIndex(units),
    Dict{Tuple{Int,Int},MSUC.Line}(), [Int[]], [DEMAND], MSUC.ARMAProcess(1; S = 1), solver())
N = instance.N
unit = units[1]

println("technical limits (MW): Pmin = $(unit.MinPower), Pmax = $(unit.MaxPower), R^U = R^D = $(unit.DeltaRampUp), S^U = S^D = $(unit.MinPower)")
println("costs: generation c = $(unit.LinearTerm) per MWh, fixed = $(unit.ConstTerm) per period on, ",
        "startup = $(unit.StartUpCost), shutdown = $(unit.StartDownCost)")
println("shedding and overproduction costs: $(MSUC.SHEDDING_COST) and $(MSUC.CURTAILEMENT_COST) per MWh; ",
        "$(length(unit.intervals)) operating intervals")
println("\nperiod t   ", join([@sprintf("%5d", t) for t in 1:T]))
println("demand     ", join([@sprintf("%5.0f", d) for d in DEMAND]))

## Exact optimum, optimal schedule and production trajectory

The extensive formulation (`multistage_extensive`) gives the exact optimum of the deterministic problem, to which both
bounds must converge. For the production trajectory of the optimal schedule, we solve the dispatch LP
with a fixed schedule, using the same capacity and ramp constraints as the code (`add_unit_dynamics!`);
we check that its cost is the exact future value given by the extensive formulation, and that the trajectory is unique.

In [ ]:
# Intervals [a, b] from the on/off status at periods 0, ..., T: on during periods a, ..., b-1
function status_intervals(status)
    res = Tuple{Int,Int}[]
    t = 0
    while t < length(status)
        if status[t + 1] > 0.5
            a = t
            while t < length(status) && status[t + 1] > 0.5
                t += 1
            end
            push!(res, (a, t))
        else
            t += 1
        end
    end
    return res
end
intervals_text(ints) = isempty(ints) ? "all off" : join(["[$a:$b]" for (a, b) in ints], " ")

# Schedule (u, v, w, selected intervals) from a list of intervals (j, a, b): u_t = sum of gamma_{a,b} with a <= t < b
function schedule_from_intervals(selected)
    u, v, w = zeros(N, T + 1), zeros(N, T), zeros(N, T)
    for (j, a, b) in selected
        for t in max(a, 0):min(b - 1, T)
            u[j, t + 1] = 1
        end
        a >= 1 && (v[j, a] = 1)
        b <= T && (w[j, b] = 1)
    end
    ids = sort([instance.intervals.id[c] for c in selected])
    return MSUC.Schedule(u, v, w, ids, [instance.intervals.unit[id] for id in ids])
end

# Exact future value V*(x): optimal dispatch cost of a schedule, from the extensive formulation with the schedule fixed
function exact_value(selected)
    gamma = Dict((u.name, a, b) => Float64((u.name, a, b) in selected) for u in units for (a, b) in u.intervals)
    res = MSUC.multistage_extensive(instance; force = FORCE, gap = 1e-9, timelimit = 60, res_gamma = gamma)
    return res[1] - MSUC.schedule_commitment_cost(instance, schedule_from_intervals(selected))
end

# Dispatch LP of the unit for a given schedule: output, shedding, overproduction, cost, and whether the output is unique
function dispatch_optimal(selected)
    p = schedule_from_intervals(selected)
    m = Model(solver())
    @variable(m, y[0:T] >= 0)
    @variable(m, shedding[1:T] >= 0)
    @variable(m, surplus[1:T] >= 0)
    fix(y[0], unit.InitialPower; force = true)
    for t in 1:T
        MSUC.add_unit_dynamics!(m, unit, y[t], y[t - 1], p.u[1, t + 1], p.u[1, t], p.v[1, t], p.w[1, t])
    end
    @constraint(m, [t in 1:T], y[t] - surplus[t] + shedding[t] == DEMAND[t])
    cost = sum(unit.LinearTerm * y[t] + MSUC.SHEDDING_COST * shedding[t] + MSUC.CURTAILEMENT_COST * surplus[t] for t in 1:T)
    @objective(m, Min, cost)
    optimize!(m)
    val = objective_value(m)
    ystar = [value(y[t]) for t in 1:T]
    res = (y = ystar, shedding = [value(shedding[t]) for t in 1:T], surplus = [value(surplus[t]) for t in 1:T],
           cost = val, unique = true)
    # Uniqueness: under the optimal-cost constraint, no output y_t can be smaller or larger
    @constraint(m, cost <= val + 1e-6)
    for t in 1:T, sense in (MOI.MIN_SENSE, MOI.MAX_SENSE)
        set_objective_function(m, y[t])
        set_objective_sense(m, sense)
        optimize!(m)
        abs(objective_value(m) - ystar[t]) > 1e-6 && return merge(res, (unique = false,))
    end
    return res
end

exact = MSUC.multistage_extensive(instance; force = FORCE, gap = 1e-9, timelimit = 60)
opt = exact[1]
optimal = [(1, a, b) for (a, b) in status_intervals([exact[4][1][1, t] for t in 0:T])]
x_opt = schedule_from_intervals(optimal)
commitment = MSUC.schedule_commitment_cost(instance, x_opt)
disp = dispatch_optimal(optimal)

@printf("exact optimum = %.1f\noptimal schedule: %s\n", opt, intervals_text([(a, b) for (_, a, b) in optimal]))
@printf("production y*_t = %s MW  (unique trajectory: %s)\n", join([@sprintf("%.0f", y) for y in disp.y], ", "), disp.unique)
@printf("unserved demand = %.0f MWh, overproduction = %.0f MWh\n", sum(disp.shedding), sum(disp.surplus))
@printf("total cost = %.0f = %.0f (commitment) + %.0f (dispatch)\n", opt, commitment, disp.cost)

# values from the paper
@assert optimal == [(1, 1, 7)] && all(isapprox.(disp.y, [40, 70, 100, 70, 100, 90]))
@assert sum(disp.shedding) == 90 && sum(disp.surplus) == 10
@assert isapprox(opt, 80200) && isapprox(commitment, 800) && isapprox(disp.cost, 79400) && disp.unique
@assert isapprox(disp.cost, exact_value(optimal))        # dispatch LP = extensive formulation with fixed schedule

## Convergence of the lower bound (Table 2)

Both algorithms start from the zero approximation. We train each formulation and record the bound `L_k` of the
root problem at iteration `k`, computed with the `k-1` cuts of the previous backward passes, together with the
root schedule `x^k`. A star marks the first iteration where the bound reaches the optimum.

In [ ]:
K = 7        # enough to see the 3-bin converge (increase if the solver changes)

# Returns the model, the bounds L_k and the schedules x^k of each iteration
function train_policy(builder, method; iterations = K, seed = 0)
    model = MSUC.PolicyGraph(builder, instance, FORCE, MSUC.Graph(T + 1), method;
                             lower_bound = 0.0, optimizer = solver(seed))
    schedules = MSUC.Schedule[]
    Random.seed!(MSUC.SEED)
    MSUC.train(model; iteration_limit = iterations, mip_gap = 1e-9, print_level = 0,
               run_numerical_stability_report = false,
               post_iteration_callback = result -> push!(schedules, model.schedule))
    bounds = [l.bound for l in model.most_recent_training_results.log]
    return (model = model, bounds = bounds, schedules = schedules)
end

# First iteration k where the bound reaches the optimum: n = k - 1 cuts suffice
first_optimal_iteration(bounds) = findfirst(b -> b >= opt - 1e-6, bounds)
cuts_needed(bounds) = (k = first_optimal_iteration(bounds); k === nothing ? nothing : k - 1)

ext = train_policy(MSUC.subproblem_builder_UC_extended, MSUC.ExtendedFormulation)
bin3 = train_policy(MSUC.subproblem_builder_UC_def, MSUC.SDDPFormulation)

function bound_row(name, bounds)
    k0 = first_optimal_iteration(bounds)
    return @sprintf("%-14s", name) * join([@sprintf("%9.0f%s", b, k == k0 ? "*" : " ") for (k, b) in enumerate(bounds)])
end
println("iteration k   ", join([@sprintf("%9d ", k) for k in 1:K]))
println(bound_row("L_k 3-bin", bin3.bounds))
println(bound_row("L_k interval", ext.bounds))
@printf("\nnumber of cuts to reach the optimum: interval %s, 3-bin %s\n",
        cuts_needed(ext.bounds), cuts_needed(bin3.bounds))

# values from the paper
@assert round.(Int, ext.bounds) == [0, 39000, 80200, 80200, 80200, 80200, 80200]
@assert round.(Int, bin3.bounds) == [0, 800, 73172, 79472, 79522, 80178, 80200]
@assert cuts_needed(ext.bounds) == 2 && cuts_needed(bin3.bounds) == 6

In [ ]:
plot(1:K, ext.bounds, label = "interval", marker = :circle, xlabel = "iteration k", ylabel = "lower bound L_k",
     title = "Lower bound", legend = :bottomright)
plot!(1:K, bin3.bounds, label = "3-bin", marker = :square)
hline!([opt], label = "optimum", linestyle = :dash)

## The first iteration: the all-off schedule

Without cuts, the root problem only minimizes the commitment cost; since fixed and startup costs are
positive, its only solution is the all-off schedule, in both formulations: the first cuts are
built at the same schedule `x^1`. With everything off, all the demand is shed, so the dispatch cost is
`700 sum_t d_t`.

A schedule is represented either by `x = (u, v, w)` or by the interval variables `gamma`: choosing
`gamma_{a,b} = 1` turns the unit on from `a` to `b-1`, so `u_t = sum_{[a:b], a <= t < b} gamma_{a,b}`.

In [ ]:
all_off = Tuple{Int,Int,Int}[]
@assert ext.schedules[1].x == bin3.schedules[1].x && all(iszero, ext.schedules[1].x)     # same schedule x^1: all off
cost_all_off = exact_value(all_off)
@printf("dispatch cost of the all-off schedule = %.0f; 700 x total demand = %.0f\n",
        cost_all_off, MSUC.SHEDDING_COST * sum(DEMAND))
@assert isapprox(cost_all_off, 385000) && isapprox(cost_all_off, MSUC.SHEDDING_COST * sum(DEMAND))

# x = x(gamma) in the code: u_t = sum of gamma_{a,b} over the selected intervals covering t, at each iteration
u_from_gamma(p) = [count(id -> instance.intervals.a[id] <= t < instance.intervals.b[id], p.selected) for t in 1:T]
@assert all(u_from_gamma(p) == p.u[1, 2:end] for p in ext.schedules)
println("x = x(gamma) verified at the ", K, " schedules of the interval formulation")

### The first root cuts

`cut^{3b,2}_1(x) = 385 000 - 68 000 sum_t u_t` and `cut^{ext,2}_1(gamma) = 385 000 + sum_{[a:b]} tau_{a,b} gamma_{a,b}`, with
`tau_{a,b} = -(700 - c) sum_{t=a}^{b-1} ybar_{a,b,t}`, where `ybar` is the maximal reachable output of the interval
(`reference_output` in the code). The 3-bin coefficient `-(700 - c) Pmax = -68 000` depends only on the length of the
interval; since `ybar <= Pmax`, `tau_{a,b} >= -68 000 (b - a)`: the first interval cut is at least as strong as
the first 3-bin cut at every admissible schedule.

In [ ]:
cut_3bin = bin3.model[1].cuts[2]          # [1]: trivial cut theta >= 0, [k+1]: cut of iteration k
cut_interval = ext.model[1].cuts[2]
price = MSUC.SHEDDING_COST - unit.LinearTerm                     # 700 - c

# coefficients of the 3-bin cut on u (periods 1, ..., T), v and w, on y_0 and on zeta_0
tu = [cut_3bin.tau[MSUC.u_index(instance, 1, t)] for t in 1:T]
tv = [cut_3bin.tau[MSUC.v_index(instance, 1, t)] for t in 1:T]
tw = [cut_3bin.tau[MSUC.w_index(instance, 1, t)] for t in 1:T]
@printf("constant: 3-bin %.0f, interval %.0f\n", cut_3bin.alpha, cut_interval.alpha)
@printf("3-bin: coefficient of u_t = %s; v and w zero: %s; y_0 and zeta_0 zero: %s\n",
        join(unique(tu), ", "), all(iszero, tv) && all(iszero, tw),
        all(iszero, cut_3bin.beta) && all(iszero, cut_3bin.delta))
@assert cut_3bin.alpha == cut_interval.alpha == 385000
@assert all(tu .== -price * unit.MaxPower) && -price * unit.MaxPower == -68000 && all(iszero, tv) && all(iszero, tw)
@assert all(iszero, cut_3bin.beta) && all(iszero, cut_3bin.delta) && all(iszero, cut_interval.delta)

ybar(a, b, t) = MSUC.reference_output(unit, a, b, t, T)                       # maximal reachable output
tau_tilde(a, b) = -price * sum(ybar(a, b, t) for t in a:(b - 1))              # coefficient of [a:b], paper formula
tau_int(a, b) = cut_interval.tau[instance.intervals.id[(1, a, b)]]              # coefficient stored in the interval cut

@assert all(isapprox(tau_int(a, b), tau_tilde(a, b)) for (a, b) in unit.intervals)
@assert all(ybar(a, b, t) <= unit.MaxPower for (a, b) in unit.intervals for t in a:(b - 1))
@assert all(tau_int(a, b) >= -price * unit.MaxPower * (b - a) - 1e-6 for (a, b) in unit.intervals)
println("coefficient of [a:b]: interval (in brackets: -68 000 (b - a), 3-bin coefficient of the same interval)")
println("      ", join([@sprintf("%19s", "b = $b") for b in 2:(T + 1)]))
for a in 1:T
    print("a = $a ")
    for b in 2:(T + 1)
        print([a, b] in unit.intervals ? @sprintf("%9.0f [%7.0f]", tau_int(a, b), -price * unit.MaxPower * (b - a)) : " "^19)
    end
    println()
end

In [ ]:
# Exhaustive check over ALL admissible schedules (sets of disjoint intervals separated by at least one off period):
# both cut formulas and the inequality between them.
function all_schedules()
    res = Vector{Vector{Tuple{Int,Int,Int}}}()
    function rec(prefix, last_end)
        push!(res, copy(prefix))
        for (a, b) in unit.intervals
            if a > last_end
                push!(prefix, (1, a, b))
                rec(prefix, b)
                pop!(prefix)
            end
        end
    end
    rec(Tuple{Int,Int,Int}[], 0)
    return res
end

# Value of a root cut at a schedule: alpha + delta' zeta_0 + (term in x) + (term in y_0)
function cut_value(cut, sched)
    A, b = MSUC.instantiate(cut, sched, N)
    return A + dot(b, [u.InitialPower for u in units]) + dot(cut.delta, instance.arma.zeta0)
end

schedules = all_schedules()
diffs = Float64[]
for selected in schedules
    p = schedule_from_intervals(selected)
    v3b, vint = cut_value(cut_3bin, p), cut_value(cut_interval, p)
    @assert isapprox(v3b, 385000 - 68000 * sum(p.u[1, 2:end]))
    @assert isapprox(vint, 385000 + sum((tau_tilde(a, b) for (_, a, b) in selected); init = 0.0))
    push!(diffs, vint - v3b)
end
@printf("%d admissible schedules: interval cut - 3-bin cut >= %.0f everywhere, strictly positive except at the empty schedule: %s\n",
        length(schedules), minimum(diffs), all(>(0), diffs[2:end]))
@assert all(>=(-1e-6), diffs)

## Additional checks (not covered in the paper)

* **Sensitivity to the Gurobi seed.** The root problem has multiple optimal solutions, so the visited schedule and
  the number of 3-bin cuts depend on which one the solver picks: we redo both trainings for four seeds.
* **Bracketing.** The lower bound never exceeds the optimum, and the cost of the trained policy (root schedule,
  then dispatch with the cuts) bounds it from above; since the uncertainty is deterministic, the three quantities coincide.

In [ ]:
println("  seed       interval cuts     3-bin cuts   same schedule at iteration 1")
for seed in 0:3
    e = train_policy(MSUC.subproblem_builder_UC_extended, MSUC.ExtendedFormulation; seed = seed)
    b = train_policy(MSUC.subproblem_builder_UC_def, MSUC.SDDPFormulation; seed = seed)
    @printf("%6d  %18s  %13s   %s\n", seed, cuts_needed(e.bounds), cuts_needed(b.bounds),
            e.schedules[1].x == b.schedules[1].x)
end

In [ ]:
for (name, r) in (("interval", ext), ("3-bin", bin3))
    cost = MSUC.simulate(r.model).cost
    @printf("%-10s bound = %.4f   optimum = %.4f   policy = %.4f\n", name, r.bounds[end], opt, cost)
    @assert r.bounds[end] <= opt + 1e-6 && cost >= opt - 1e-6
end